# Lexel Stage 1: QLoRA SFT on Qwen2.5-1.5B-Instruct (Kaggle T4 / Unsloth Studio)

This self-contained notebook trains the Stage 1 Supervised Fine-Tuning (SFT) adapter for **Lexel** using Unsloth 4-bit QLoRA (with automatic fallback to standard HuggingFace `transformers` + `peft` + `trl` if `unsloth` is not installed).

## What This Notebook Fixes for Kaggle
1. **Dynamic Package Installation**: Uses `subprocess.check_call` to install `unsloth` cleanly on Kaggle's current PyTorch/CUDA runtime without hardcoded `xformers` ABI clashes, and disables `wandb` prompts automatically.
2. **Zero-Config Dataset Discovery**: Automatically locates `lexel_sft_train.jsonl` and `lexel_sft_val.jsonl` anywhere under `/kaggle/input` or local `ml/datasets/` (and auto-generates a Crucible-verified fallback split if no dataset is attached yet).
3. **TRL Signature Compatibility (`processing_class` vs `tokenizer`, `max_length` vs `max_seq_length`)**: Inspects installed `trl.SFTConfig` and `trl.SFTTrainer` with `inspect.signature` so it runs across all 2025-2026 TRL releases without `TypeError`.
4. **Safe Completion-Only Masking + Sequence Packing**: Applies `train_on_responses_only` before enabling packing (or pre-masks `-100` prompt tokens directly) and audits a real batch to verify supervised completion tokens before `.train()` starts.

In [ ]:
# Cell 1: Environment Setup & Kaggle Package Installation
import importlib.util
import os
import subprocess
import sys

os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Pin to GPU 0 by default on Kaggle 2x T4 so 4-bit bitsandbytes QLoRA never hits DataParallel device-mismatch errors
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ and "CUDA_VISIBLE_DEVICES" not in os.environ:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

def ensure_training_packages() -> None:
    required = ("trl", "peft", "datasets", "accelerate", "bitsandbytes")
    missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]
    has_unsloth = importlib.util.find_spec("unsloth") is not None

    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ and (missing or not has_unsloth):
        print("Installing Unsloth + QLoRA training dependencies for Kaggle GPU...")
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--upgrade",
                "unsloth",
                "unsloth_zoo",
                "trl",
                "peft",
                "accelerate",
                "bitsandbytes",
                "datasets",
            ]
        )
    elif missing:
        print(f"Installing missing packages: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

ensure_training_packages()

import torch
print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Configuration & Automatic Dataset Discovery
import hashlib
import inspect
import json
import time
from datetime import UTC, datetime
from pathlib import Path

# Import unsloth first if installed so its kernel optimizations patch transformers/trl cleanly
try:
    import unsloth
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from unsloth.chat_templates import get_chat_template, train_on_responses_only
    HAS_UNSLOTH = True
except Exception as exc:
    print(f"Unsloth fast path unavailable ({exc}); using standard Transformers + PEFT + TRL fallback.")
    HAS_UNSLOTH = False
    def is_bfloat16_supported() -> bool:
        return torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

MODEL_TIER = os.environ.get("LEXEL_MODEL_TIER", "lexel-1.5b")
UNSLOTH_MODEL_MAP = {
    "lexel-1.5b": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    "lexel-0.5b": "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
}
HF_FALLBACK_MODEL_MAP = {
    "lexel-1.5b": "Qwen/Qwen2.5-1.5B-Instruct",
    "lexel-0.5b": "Qwen/Qwen2.5-0.5B-Instruct",
}
BASE_MODEL_ID = UNSLOTH_MODEL_MAP[MODEL_TIER] if HAS_UNSLOTH else HF_FALLBACK_MODEL_MAP[MODEL_TIER]

MAX_SEQ_LENGTH = 1536
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0
TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

def resolve_data_dir() -> Path:
    candidates = [
        Path(os.environ["LEXEL_DATA_DIR"]) if "LEXEL_DATA_DIR" in os.environ else None,
        Path("/kaggle/input/lexel-crucible-datasets"),
        Path("/kaggle/input/datasets/lawandsoran/lexel-crucible-datasets"),
        Path.cwd() / "ml" / "datasets",
        Path.cwd().parent / "datasets",
        Path.cwd() / "datasets",
    ]
    for cand in candidates:
        if cand and (cand / "lexel_sft_train.jsonl").is_file():
            return cand.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.is_dir():
        matches = sorted(kaggle_input.rglob("lexel_sft_train.jsonl"))
        if matches:
            return matches[0].parent.resolve()

    # Fallback: if running inside the Lexel repo, build datasets on the fly
    for root_cand in (Path.cwd(), *Path.cwd().parents):
        if (root_cand / "ml" / "crucible" / "dataset_builder.py").is_file():
            if str(root_cand) not in sys.path:
                sys.path.insert(0, str(root_cand))
            from ml.crucible.dataset_builder import build_all_datasets
            out_dir = root_cand / "ml" / "datasets"
            build_all_datasets(out_dir)
            return out_dir.resolve()

    # Self-contained Kaggle emergency fallback if the user forgot to attach the dataset
    fallback_dir = Path("/kaggle/working/lexel-datasets") if Path("/kaggle/working").is_dir() else (Path.cwd() / "lexel-datasets")
    fallback_dir.mkdir(parents=True, exist_ok=True)
    train_p = fallback_dir / "lexel_sft_train.jsonl"
    val_p = fallback_dir / "lexel_sft_val.jsonl"
    if not train_p.is_file():
        print("WARNING: No attached Kaggle dataset found; writing built-in Crucible seed split to", fallback_dir)
        seed_rows = [
            {
                "id": f"seed_{i:03d}",
                "surface_id": "web_github_pr",
                "task": "deslop",
                "messages": [
                    {"role": "system", "content": "You are Lexel, a precise human writing editor.\nRules:\n- Output ONLY the rewritten text. No preamble, no quotes, no explanation.\n- Preserve every number, metric, ID (#42), URL, and `code` token exactly.\n- Never use em dashes (\u2014) or AI buzzwords (delve, tapestry, seamless, leverage, game-changer, unlock, pivotal).\n- Surface: GitHub (web_github_pr)\n- Max characters: 4000\n- Task: Remove AI buzzwords, corporate padding, filler openers, and em dashes. Make every sentence direct and human."},
                    {"role": "user", "content": f"<source>\nAwesome work! This PR seamlessly leverages `extract_unit_bound_metrics` in `verifier.rs` to fix #{40 + i} and keep latency at 0.4ms.\n</source>"},
                    {"role": "assistant", "content": f"Uses `extract_unit_bound_metrics` in `verifier.rs` to fix #{40 + i} and keep check latency at 0.4ms."},
                ],
            }
            for i in range(24)
        ]
        train_p.write_text("\n".join(json.dumps(r) for r in seed_rows[:20]) + "\n", encoding="utf-8")
        val_p.write_text("\n".join(json.dumps(r) for r in seed_rows[20:]) + "\n", encoding="utf-8")
    return fallback_dir.resolve()

DATA_DIR = resolve_data_dir()
TRAIN_FILE = DATA_DIR / "lexel_sft_train.jsonl"
VAL_FILE = DATA_DIR / "lexel_sft_val.jsonl"
OUTPUT_DIR = Path(os.environ.get("LEXEL_SFT_OUT", f"./outputs/{MODEL_TIER}-sft-adapter")).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Selected Tier: {MODEL_TIER} -> {BASE_MODEL_ID}")
print(f"Resolved DATA_DIR: {DATA_DIR}")
print(f"Train file: {TRAIN_FILE} ({TRAIN_FILE.stat().st_size} bytes)")
print(f"Val file:   {VAL_FILE} ({VAL_FILE.stat().st_size} bytes)")
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
# Cell 3: Load 4-bit Base Model & Attach LoRA Adapters
if HAS_UNSLOTH:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    tokenizer = get_chat_template(
        tokenizer,
        chat_template="qwen-2.5",
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        target_modules=TARGET_MODULES,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
        use_rslora=False,
        loftq_config=None,
    )
else:
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if is_bfloat16_supported() else torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config if torch.cuda.is_available() else None,
        device_map="auto" if torch.cuda.is_available() else None,
    )
    if torch.cuda.is_available():
        model = prepare_model_for_kbit_training(model)
    peft_cfg = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES,
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, peft_cfg)

tokenizer.padding_side = "right"
model.print_trainable_parameters()

In [ ]:
# Cell 4: Load Crucible-Verified Datasets & Format Qwen-2.5 ChatML Prompts
dataset = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_FILE),
        "validation": str(VAL_FILE),
    },
)

def format_chat_prompts(batch):
    texts = [
        tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        for msgs in batch["messages"]
    ]
    return {"text": texts}

dataset = dataset.map(
    format_chat_prompts,
    batched=True,
    remove_columns=[c for c in dataset["train"].column_names if c != "text"],
)
print(f"Train rows: {len(dataset['train'])}, Validation rows: {len(dataset['validation'])}")
print("\nSample formatted ChatML prompt:\n", dataset["train"][0]["text"][:600])

In [ ]:
# Cell 5: Configure SFTTrainer with Version-Safe kwargs & Verify Supervised Labels
sft_cfg_params = inspect.signature(SFTConfig.__init__).parameters
sft_trainer_params = inspect.signature(SFTTrainer.__init__).parameters

cfg_kwargs = {
    "output_dir": str(OUTPUT_DIR),
    "per_device_train_batch_size": 2,
    "per_device_eval_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.05,
    "num_train_epochs": 2,
    "optim": "adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    "weight_decay": 0.01,
    "fp16": bool(torch.cuda.is_available() and not is_bfloat16_supported()),
    "bf16": bool(torch.cuda.is_available() and is_bfloat16_supported()),
    "logging_steps": 5,
    "save_strategy": "epoch",
    "seed": 42,
    "report_to": "none",
}

if "eval_strategy" in sft_cfg_params:
    cfg_kwargs["eval_strategy"] = "epoch"
elif "evaluation_strategy" in sft_cfg_params:
    cfg_kwargs["evaluation_strategy"] = "epoch"

if "dataset_text_field" in sft_cfg_params:
    cfg_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in sft_cfg_params:
    cfg_kwargs["max_seq_length"] = MAX_SEQ_LENGTH
elif "max_length" in sft_cfg_params:
    cfg_kwargs["max_length"] = MAX_SEQ_LENGTH

# Initialize with packing=False first so train_on_responses_only can safely map token boundaries
if "packing" in sft_cfg_params:
    cfg_kwargs["packing"] = False
if "padding_free" in sft_cfg_params:
    cfg_kwargs["padding_free"] = False

sft_config = SFTConfig(**cfg_kwargs)

trainer_kwargs = {
    "model": model,
    "train_dataset": dataset["train"],
    "eval_dataset": dataset["validation"],
    "args": sft_config,
}
if "processing_class" in sft_trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in sft_trainer_params:
    trainer_kwargs["tokenizer"] = tokenizer
if "dataset_text_field" in sft_trainer_params and "dataset_text_field" not in cfg_kwargs:
    trainer_kwargs["dataset_text_field"] = "text"
if "max_seq_length" in sft_trainer_params and "max_seq_length" not in cfg_kwargs and "max_length" not in cfg_kwargs:
    trainer_kwargs["max_seq_length"] = MAX_SEQ_LENGTH

trainer = SFTTrainer(**trainer_kwargs)

# Apply strict completion-only loss masking (-100 on all system/user prompt tokens)
if HAS_UNSLOTH:
    try:
        trainer = train_on_responses_only(
            trainer,
            instruction_part="<|im_start|>user\n",
            response_part="<|im_start|>assistant\n",
        )
        print("Applied Unsloth train_on_responses_only completion mask.")
    except Exception as mask_exc:
        print(f"Note: train_on_responses_only fallback ({mask_exc}); continuing with standard SFT collator.")

# Audit one real batch from train_dataloader to guarantee finite supervised tokens
sample_batch = next(iter(trainer.get_train_dataloader()))
supervised_count = int((sample_batch["labels"] != -100).sum().item())
assert supervised_count > 0, "CRITICAL: Zero supervised tokens in batch labels!"
print(f"Batch label audit PASSED: {supervised_count} supervised completion tokens in first batch (shape={tuple(sample_batch['input_ids'].shape)}).")

In [ ]:
# Cell 6: Execute Stage 1 Training & Save SFT Adapter + run_manifest.json
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()
train_stats = trainer.train()
elapsed_sec = time.perf_counter() - t0

peak_vram_gb = (
    round(torch.cuda.max_memory_allocated() / (1024**3), 3)
    if torch.cuda.is_available()
    else 0.0
)

adapter_dir = OUTPUT_DIR / "adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

train_sha256 = hashlib.sha256(TRAIN_FILE.read_bytes()).hexdigest()
val_sha256 = hashlib.sha256(VAL_FILE.read_bytes()).hexdigest()

run_manifest = {
    "stage": "stage1_sft",
    "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "model_tier": MODEL_TIER,
    "base_model_id": BASE_MODEL_ID,
    "max_seq_length": MAX_SEQ_LENGTH,
    "lora": {
        "r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "target_modules": TARGET_MODULES,
        "use_gradient_checkpointing": "unsloth" if HAS_UNSLOTH else True,
        "adapter_path": str(adapter_dir),
    },
    "hyperparameters": {
        "learning_rate": 2e-4,
        "lr_scheduler_type": "cosine",
        "warmup_ratio": 0.05,
        "num_train_epochs": 2,
        "optim": cfg_kwargs["optim"],
        "train_on_responses_only": True,
    },
    "datasets": {
        "train_file": str(TRAIN_FILE),
        "train_rows": len(dataset["train"]),
        "train_sha256": train_sha256,
        "val_file": str(VAL_FILE),
        "val_rows": len(dataset["validation"]),
        "val_sha256": val_sha256,
    },
    "telemetry": {
        "global_step": int(trainer.state.global_step),
        "training_loss": round(float(train_stats.training_loss), 6),
        "wall_time_sec": round(elapsed_sec, 2),
        "peak_vram_gb": peak_vram_gb,
    },
}

manifest_path = OUTPUT_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(run_manifest, indent=2) + "\n", encoding="utf-8")
print("Saved Stage 1 SFT adapter and run_manifest.json to:", OUTPUT_DIR)
print(json.dumps(run_manifest["telemetry"], indent=2))